In [ ]:
from typing import List

import torch
from at2v.dloader import TagDataset
from at2v.shared import get_setup
from torch.utils.data import DataLoader
import torch.nn.functional as F

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data, tagtok, anitag2vec = get_setup(
    device=device,
    prefix_path= ".."
)

anitag2vec.load_state_dict(torch.load("../checkpoints/anitag2vec_e15_s25000_p905216.pth"))
anitag2vec.to(device)
anitag2vec.eval()

In [ ]:
def to_dataloader(inputs: List[List[str]]):
    dataset = TagDataset(
        list_of_tags=inputs,
        max_len_cut=64,
        tokenizer=tagtok
    )
    return DataLoader(dataset, batch_size=len(inputs), shuffle=False)

def run_inference(inputs: List[List[str]]):
    batches = to_dataloader(inputs)
    for batch in batches:
        batch = batch.to(device)
        return anitag2vec(batch)

def rank_cosim(query: List[str], items: List[List[str]]):
    q = F.normalize(run_inference([query]), dim=1)   # (1, O)
    xs = F.normalize(run_inference(items), dim=1)    # (N, O)
    scores = (q @ xs.T).squeeze(0)                   # (N,)

    indices = torch.argsort(scores, descending=True)
    ranked_items = [items[i] for i in indices.tolist()]

    return list(zip(scores[indices], ranked_items))

In [ ]:
query = data.real_examples[600]
ranked = rank_cosim(
    query,
    data.real_examples[2000:2500]
)

In [ ]:
print("Closest to:", ", ".join(query))
for index, (cosim, tags) in enumerate(ranked[:3]):
    perc = (1 + cosim.item()) / 2
    print(f"# {index + 1}, {round(perc * 100 * 100) / 100}%: {', '.join(sorted(tags))}")